# H&M 1st Stage Candidate Generation Demo

This lightweight notebook demonstrates the first stage of a two-stage recommender pipeline:

1. Load a recent slice of the H&M transactions with Polars.
2. Encode `customer_id` and `article_id` to integer keys.
3. Generate candidates from history, global popularity, age-bucket popularity, and category popularity.
4. Rank candidates with a weighted linear `candidate_score`.
5. Measure Recall@k and MAP@k at `k = 12, 24, 48, 96, 120`.
6. Plot recall customer mean and MAP by cutoff.


In [ ]:
try:
    import polars as pl
except ImportError:
    %pip -q install 'polars[gpu]'
    import polars as pl

try:
    import matplotlib.pyplot as plt
except ImportError:
    %pip -q install matplotlib
    import matplotlib.pyplot as plt

import math
from collections import defaultdict
from datetime import date, timedelta
from pathlib import Path

# Polars GPU engine is used for LazyFrame materialization when available.
# On CPU-only Kaggle sessions this falls back to the normal Polars engine.
USE_POLARS_GPU = True


def collect_pl(lf: pl.LazyFrame) -> pl.DataFrame:
    global USE_POLARS_GPU
    if USE_POLARS_GPU:
        try:
            return lf.collect(engine="gpu")
        except Exception as exc:
            USE_POLARS_GPU = False
            print(f"Polars GPU unavailable; falling back to CPU. Reason: {type(exc).__name__}: {exc}")
    return lf.collect()


pl.__version__

In [ ]:
# Kaggle path first, local repo path second.
KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations")
LOCAL_DATA_DIR = Path("../data/raw")
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR

TRANSACTIONS_FILE = DATA_DIR / "transactions_train.csv"
CUSTOMERS_FILE = DATA_DIR / "customers.csv"
ARTICLES_FILE = DATA_DIR / "articles.csv"

assert TRANSACTIONS_FILE.exists(), f"Missing {TRANSACTIONS_FILE}"
assert CUSTOMERS_FILE.exists(), f"Missing {CUSTOMERS_FILE}"
assert ARTICLES_FILE.exists(), f"Missing {ARTICLES_FILE}"

# Teaching/demo parameters. Set MAX_VALID_CUSTOMERS=None for a fuller run.
VALIDATION_CUTOFF = date(2020, 9, 8)
LABEL_DAYS = 7
EVAL_CUTOFFS = [12, 24, 48, 96, 120]
MAX_VALID_CUSTOMERS = 5_000

CANDIDATE_LIMIT = 120
HISTORY_DAYS = 45
HISTORY_TOP_N = 24
GLOBAL_POPULARITY_DAYS = [7, 14, 30, 60]
GLOBAL_TOP_N = 50
AGE_POPULARITY_DAYS = 30
AGE_TOP_N = 40
CATEGORY_HISTORY_DAYS = 120
CATEGORY_POPULARITY_DAYS = 45
CATEGORY_TOP_N = 16
CATEGORY_PREF_TOP_N = 2
CATEGORY_COLS_RAW = ["section_name", "garment_group_name", "index_group_name"]

SOURCE_WEIGHTS = {
    "history": 4.0,
    "category": 2.0,
    "global7": 2.0,
    "global14": 1.4,
    "global30": 0.9,
    "global60": 0.5,
    "age14": 0.8,
}
RANK_WEIGHT = 2.0
COUNT_WEIGHT = 0.25
RECENCY_WEIGHT = 3.0

LABEL_END = VALIDATION_CUTOFF + timedelta(days=LABEL_DAYS)
EARLIEST_NEEDED_DATE = VALIDATION_CUTOFF - timedelta(days=max(HISTORY_DAYS, max(GLOBAL_POPULARITY_DAYS), AGE_POPULARITY_DAYS, CATEGORY_HISTORY_DAYS, CATEGORY_POPULARITY_DAYS))

DATA_DIR, EARLIEST_NEEDED_DATE, VALIDATION_CUTOFF, LABEL_END

## Load and encode a recent data slice

For the production repo we prebuild shared encoded parquet files. In this standalone Kaggle demo, we build a small in-notebook encoding from raw CSV so students can see the full flow.

In [ ]:
transactions_raw = collect_pl(
    pl.scan_csv(
        TRANSACTIONS_FILE,
        schema_overrides={"t_dat": pl.Utf8, "customer_id": pl.Utf8, "article_id": pl.Utf8},
    )
    .select(["t_dat", "customer_id", "article_id"])
    .with_columns(pl.col("t_dat").str.strptime(pl.Date, strict=False))
    .filter((pl.col("t_dat") > pl.lit(EARLIEST_NEEDED_DATE)) & (pl.col("t_dat") <= pl.lit(LABEL_END)))
)

customers_raw = collect_pl(
    pl.scan_csv(
        CUSTOMERS_FILE,
        schema_overrides={"customer_id": pl.Utf8, "age": pl.Float32},
    ).select(["customer_id", "age"])
)

articles_raw = collect_pl(
    pl.scan_csv(
        ARTICLES_FILE,
        schema_overrides={"article_id": pl.Utf8, **{col: pl.Utf8 for col in CATEGORY_COLS_RAW}},
    ).select(["article_id", *CATEGORY_COLS_RAW])
)

customer_map = collect_pl(
    pl.concat([transactions_raw.select("customer_id"), customers_raw.select("customer_id")])
    .lazy()
    .unique()
    .sort("customer_id")
    .with_row_index("customer_idx", offset=1)
    .select(["customer_idx", "customer_id"])
)
article_map = collect_pl(
    pl.concat([transactions_raw.select("article_id"), articles_raw.select("article_id")])
    .lazy()
    .unique()
    .sort("article_id")
    .with_row_index("article_idx", offset=1)
    .select(["article_idx", "article_id"])
)

transactions = collect_pl(
    transactions_raw
    .lazy()
    .join(customer_map.lazy(), on="customer_id", how="left")
    .join(article_map.lazy(), on="article_id", how="left")
    .select(["t_dat", "customer_idx", "article_idx"])
)
customers = collect_pl(
    customers_raw
    .lazy()
    .join(customer_map.lazy(), on="customer_id", how="left")
    .select(["customer_idx", "age"])
)
articles = collect_pl(articles_raw.lazy().join(article_map.lazy(), on="article_id", how="left").select(["article_idx", *CATEGORY_COLS_RAW]))

for col in CATEGORY_COLS_RAW:
    map_df = collect_pl(articles.lazy().select(col).fill_null("__missing__").unique().sort(col).with_row_index(f"{col}_idx", offset=1))
    articles = collect_pl(articles.lazy().with_columns(pl.col(col).fill_null("__missing__")).join(map_df.lazy(), on=col, how="left").drop(col))

CATEGORY_COLS = [f"{col}_idx" for col in CATEGORY_COLS_RAW]

transactions.shape, customers.shape, articles.shape

In [ ]:
def add_age_bucket(customers_df: pl.DataFrame) -> pl.DataFrame:
    return collect_pl(customers_df.lazy().with_columns(
        pl.when(pl.col("age").is_null())
        .then(pl.lit("unknown"))
        .when(pl.col("age") < 20)
        .then(pl.lit("<20"))
        .when(pl.col("age") < 30)
        .then(pl.lit("20s"))
        .when(pl.col("age") < 40)
        .then(pl.lit("30s"))
        .when(pl.col("age") < 50)
        .then(pl.lit("40s"))
        .when(pl.col("age") < 60)
        .then(pl.lit("50s"))
        .otherwise(pl.lit("60+"))
        .alias("age_bucket")
    ))


def sorted_count_frame(df: pl.DataFrame, group_cols: list[str]) -> pl.DataFrame:
    return collect_pl(
        df.lazy()
        .group_by(group_cols)
        .agg(pl.len().alias("cnt"))
        .sort(group_cols[:-1] + ["cnt", group_cols[-1]], descending=[False] * (len(group_cols) - 1) + [True, False])
    )


def actuals_for_window(transactions_df: pl.DataFrame, cutoff: date, label_days: int) -> dict[int, list[int]]:
    label_end = cutoff + timedelta(days=label_days)
    grouped = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > cutoff) & (pl.col("t_dat") <= label_end))
        .group_by("customer_idx")
        .agg(pl.col("article_idx").unique().alias("actual"))
    )
    return {int(row["customer_idx"]): [int(x) for x in row["actual"]] for row in grouped.iter_rows(named=True)}


def build_top_list(transactions_df: pl.DataFrame, cutoff: date, days: int, limit: int) -> list[tuple[int, int]]:
    start = cutoff - timedelta(days=days)
    counts = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > start) & (pl.col("t_dat") <= cutoff))
        .group_by("article_idx")
        .agg(pl.len().alias("cnt"))
        .sort(["cnt", "article_idx"], descending=[True, False])
        .head(limit)
    )
    return [(int(row["article_idx"]), int(row["cnt"])) for row in counts.iter_rows(named=True)]


def build_customer_history(transactions_df: pl.DataFrame, cutoff: date, days: int, top_n: int) -> dict[int, list[tuple[int, int, int]]]:
    start = cutoff - timedelta(days=days)
    latest = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > start) & (pl.col("t_dat") <= cutoff))
        .group_by(["customer_idx", "article_idx"])
        .agg(pl.len().alias("cnt"), pl.max("t_dat").alias("last_date"))
        .sort(["customer_idx", "last_date", "article_idx"], descending=[False, True, False])
    )
    history = defaultdict(list)
    for row in latest.iter_rows(named=True):
        customer_idx = int(row["customer_idx"])
        if len(history[customer_idx]) < top_n:
            history[customer_idx].append((int(row["article_idx"]), int(row["cnt"]), (cutoff - row["last_date"]).days))
    return dict(history)

In [ ]:
def build_age_top(transactions_df, customers_df, cutoff, days, top_n):
    start = cutoff - timedelta(days=days)
    recent = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > start) & (pl.col("t_dat") <= cutoff))
        .select(["customer_idx", "article_idx"])
        .join(customers_df.lazy().select(["customer_idx", "age_bucket"]), on="customer_idx", how="left")
        .with_columns(pl.col("age_bucket").fill_null("unknown"))
    )
    counts = sorted_count_frame(recent, ["age_bucket", "article_idx"])
    out = defaultdict(list)
    for row in counts.iter_rows(named=True):
        bucket = str(row["age_bucket"])
        if len(out[bucket]) < top_n:
            out[bucket].append((int(row["article_idx"]), int(row["cnt"])))
    return dict(out)


def build_category_top(transactions_df, articles_df, cutoff, days, category_cols, top_n):
    start = cutoff - timedelta(days=days)
    recent = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > start) & (pl.col("t_dat") <= cutoff))
        .select(["article_idx"])
        .join(articles_df.lazy(), on="article_idx", how="left")
    )
    category_top = {}
    for col in category_cols:
        counts = sorted_count_frame(collect_pl(recent.lazy().filter(pl.col(col).is_not_null()).select([col, "article_idx"])), [col, "article_idx"])
        top_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            category = int(row[col])
            if len(top_for_col[category]) < top_n:
                top_for_col[category].append((int(row["article_idx"]), int(row["cnt"])))
        category_top[col] = dict(top_for_col)
    return category_top


def build_category_preferences(transactions_df, articles_df, cutoff, days, category_cols, pref_top_n):
    start = cutoff - timedelta(days=days)
    recent = collect_pl(
        transactions_df
        .lazy()
        .filter((pl.col("t_dat") > start) & (pl.col("t_dat") <= cutoff))
        .select(["customer_idx", "article_idx", "t_dat"])
        .join(articles_df.lazy(), on="article_idx", how="left")
    )
    preferences = {}
    for col in category_cols:
        counts = collect_pl(
            recent
            .lazy()
            .filter(pl.col(col).is_not_null())
            .group_by(["customer_idx", col])
            .agg(pl.len().alias("cnt"), pl.max("t_dat").alias("last_date"))
            .sort(["customer_idx", "cnt", "last_date", col], descending=[False, True, True, False])
        )
        pref_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            customer_idx = int(row["customer_idx"])
            if len(pref_for_col[customer_idx]) < pref_top_n:
                pref_for_col[customer_idx].append(int(row[col]))
        preferences[col] = dict(pref_for_col)
    return preferences

In [ ]:
customers = add_age_bucket(customers)
actuals_all = actuals_for_window(transactions, VALIDATION_CUTOFF, LABEL_DAYS)
valid_customers = sorted(actuals_all)
if MAX_VALID_CUSTOMERS is not None:
    valid_customers = valid_customers[:MAX_VALID_CUSTOMERS]
actuals = {customer_idx: actuals_all[customer_idx] for customer_idx in valid_customers}

components = {
    "history": build_customer_history(transactions, VALIDATION_CUTOFF, HISTORY_DAYS, HISTORY_TOP_N),
    "global_top": {days: build_top_list(transactions, VALIDATION_CUTOFF, days, GLOBAL_TOP_N) for days in GLOBAL_POPULARITY_DAYS},
    "age_top": build_age_top(transactions, customers, VALIDATION_CUTOFF, AGE_POPULARITY_DAYS, AGE_TOP_N),
    "category_top": build_category_top(transactions, articles, VALIDATION_CUTOFF, CATEGORY_POPULARITY_DAYS, CATEGORY_COLS, CATEGORY_TOP_N),
    "category_preferences": build_category_preferences(transactions, articles, VALIDATION_CUTOFF, CATEGORY_HISTORY_DAYS, CATEGORY_COLS, CATEGORY_PREF_TOP_N),
    "customer_age_bucket": dict(zip(customers["customer_idx"].to_list(), customers["age_bucket"].fill_null("unknown").to_list())),
}

len(valid_customers), len(actuals_all)

In [ ]:
SOURCE_COLUMNS = ["source_history", "source_age14", "source_global7", "source_global14", "source_global30", "source_global60", "source_category"]
RANK_COLUMNS = ["history_rank", "age_rank", "global7_rank", "global14_rank", "global30_rank", "global60_rank", "category_rank"]
MISSING_RANK = 999


def new_candidate_features():
    return {
        "candidate_score": 0.0,
        **{col: 0 for col in SOURCE_COLUMNS},
        **{col: MISSING_RANK for col in RANK_COLUMNS},
    }


def score_piece(source, rank, count, recency_days=None):
    score = SOURCE_WEIGHTS[source] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * math.log1p(max(count, 0))
    if recency_days is not None:
        score += RECENCY_WEIGHT / (1.0 + max(recency_days, 0))
    return score


def add_candidate(candidates, article_idx, source, rank, count, recency_days=None):
    if article_idx not in candidates:
        if len(candidates) >= CANDIDATE_LIMIT:
            return
        candidates[article_idx] = new_candidate_features()
    row = candidates[article_idx]
    row[f"source_{source}"] = 1
    rank_col = "age_rank" if source == "age14" else f"{source}_rank"
    row[rank_col] = min(row[rank_col], rank)
    row["candidate_score"] += score_piece(source, rank, count, recency_days)


def candidate_map_for_customer(customer_idx):
    candidates = {}
    for rank, (article_idx, count, recency_days) in enumerate(components["history"].get(customer_idx, []), start=1):
        add_candidate(candidates, article_idx, "history", rank, count, recency_days)
    bucket = components["customer_age_bucket"].get(customer_idx, "unknown")
    for rank, (article_idx, count) in enumerate(components["age_top"].get(bucket, []), start=1):
        add_candidate(candidates, article_idx, "age14", rank, count)
    for rank, (article_idx, count) in enumerate(components["global_top"].get(7, []), start=1):
        add_candidate(candidates, article_idx, "global7", rank, count)
    for col in CATEGORY_COLS:
        for category in components["category_preferences"][col].get(customer_idx, []):
            for rank, (article_idx, count) in enumerate(components["category_top"][col].get(category, []), start=1):
                add_candidate(candidates, article_idx, "category", rank, count)
    for days, source in [(14, "global14"), (30, "global30"), (60, "global60")]:
        for rank, (article_idx, count) in enumerate(components["global_top"].get(days, []), start=1):
            add_candidate(candidates, article_idx, source, rank, count)
    return candidates


def ordered_candidates(candidates):
    return sorted(candidates.items(), key=lambda item: (-item[1]["candidate_score"], min(item[1][col] for col in RANK_COLUMNS), item[0]))

In [ ]:
def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_idx in enumerate(predicted[:k], start=1):
        if article_idx in seen:
            continue
        seen.add(article_idx)
        if article_idx in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)


metric_acc = {k: {"customer_recall": [], "map": []} for k in EVAL_CUTOFFS}
candidate_rows = []

for customer_idx in valid_customers:
    candidates = candidate_map_for_customer(customer_idx)
    ordered_items = ordered_candidates(candidates)
    prediction = [article_idx for article_idx, _ in ordered_items]
    actual_set = set(actuals[customer_idx])

    for k in EVAL_CUTOFFS:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]["customer_recall"].append(hits / len(actual_set) if actual_set else 0.0)
        metric_acc[k]["map"].append(apk(actual_set, prediction, k))

    for order, (article_idx, features) in enumerate(ordered_items, start=1):
        candidate_rows.append({
            "customer_idx": customer_idx,
            "article_idx": article_idx,
            "candidate_order": order,
            "candidate_score": features["candidate_score"],
            "label": int(article_idx in actual_set),
        })

metrics = pl.DataFrame([
    {
        "k": k,
        "recall_customer_mean": sum(metric_acc[k]["customer_recall"]) / len(metric_acc[k]["customer_recall"]),
        "map": sum(metric_acc[k]["map"]) / len(metric_acc[k]["map"]),
    }
    for k in EVAL_CUTOFFS
])
candidate_df = pl.DataFrame(candidate_rows)

metrics

In [ ]:
ks = metrics["k"].to_list()
recall_values = metrics["recall_customer_mean"].to_list()
map_values = metrics["map"].to_list()

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(ks, recall_values, marker="o", linewidth=2, label="Recall customer mean", color="tab:blue")
ax1.set_xlabel("k")
ax1.set_ylabel("Recall customer mean", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_xticks(ks)
ax1.grid(True, alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(ks, map_values, marker="s", linewidth=2, label="MAP", color="tab:orange")
ax2.set_ylabel("MAP", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

fig.suptitle("1st-stage candidate quality by cutoff")
fig.tight_layout()
plt.show()

In [ ]:
candidate_df.shape, candidate_df.head(10)

## What students should notice

- Recall rises as `k` increases because the candidate set is wider.
- MAP is sensitive to ordering, so `candidate_score` matters even before the 2nd-stage reranker.
- The 1st stage does not need to be a heavy model. A transparent weighted linear score is enough to teach the candidate-generation idea.
- The 2nd stage should receive this fixed candidate set and only rerank it.